# pivot-parrot in Jupyter 🦜

Interactive pivot table on the Titanic dataset — all computation runs in your browser via DuckDB WASM.

In [1]:
import pandas as pd
from jupyter_pivot_parrot import PivotParrot

df = pd.read_csv("../streamlit-pivot-parrot/titanic.csv")
df.head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [2]:
w = PivotParrot(
    data=df,
    rows=["Sex", "Pclass"],
    values=[
        {"field": "Survived", "aggregation": "avg"},
        {"field": "Fare", "aggregation": "avg"},
    ],
    height=700,
)
w

Drag fields around above, then read the current layout back in Python:

In [3]:
w.config

{'rows': ['Sex', 'Pclass'],
 'columns': [],
 'values': [{'field': 'Survived', 'aggregation': 'avg'},
  {'field': 'Fare', 'aggregation': 'avg'}],
 'filters': [],
 'customMetrics': [],
 'customFilters': [],
 'customGroupings': []}

## Remote mode

For data too big for the browser, pass a `connection` instead of a DataFrame — SQL runs on your side (warehouse, DuckDB, anything with `execute_query(sql)`), only aggregates travel to the widget:

In [4]:
import duckdb


class DuckDBConnection:
    """Any object with execute_query(sql) -> DataFrame works as a connection."""

    def __init__(self, df):
        self._con = duckdb.connect()
        self._con.register("titanic", df)

    def execute_query(self, sql, timeout=None):
        return self._con.execute(sql).df()


wr = PivotParrot(
    connection=DuckDBConnection(df),
    query="SELECT * FROM titanic",
    rows=["Embarked"],
    values=[{"field": "Fare", "aggregation": "avg"}],
    height=600,
)
wr